# JEV 3SAT - interferência em merge (Java) no Colab
Classifica cenários (base/left/right/merge .java) como **interference / not_interference** e a causa
(**different_superior_triangle / different_lateral_triangle / equal_superior_triangle / none_above**), com chamadas tipadas `client.systemOne` + `choice` do Jev.
Compara os prompts `v1_faithful` (3 assunções do custom.md) e `v2_optimized` (+ diffs + variáveis; foco em F1 e recall).

**Antes:** em *Secrets* (chave à esquerda) crie `TYPESAFE_API_KEY` e, se os repositórios forem privados, `GITHUB_TOKEN`.

In [ ]:
# 1) Node 20 + clonar repositórios (o código Java é lido como arquivos; nada é compilado/executado)
!apt-get -qq install -y nodejs npm > /dev/null 2>&1; node -v || true
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1 && apt-get -qq install -y nodejs > /dev/null 2>&1; node -v

import os
from google.colab import userdata
os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")
try: tok = userdata.get("GITHUB_TOKEN") + "@"
except Exception: tok = ""
ORG = f"https://{tok}github.com/IN0977-TAES-I"
for r in ["jev-evaluator", "interference_reduced_mergedataset", "no-interference_reduced_mergedataset"]:
    !test -d {r} || git clone -q {ORG}/{r}.git
!cd jev-evaluator && npm ci --silent

In [ ]:
# 2) Rodar os dois prompts nos dois datasets (use --limit 3 para um teste barato; --distinct = 1 pasta por número de cenário)
import subprocess
def run(variant, extra=""):
    !cd jev-evaluator && npx tsx run_3sat.ts --variant {variant} --datasets-dir .. --results-dir ../results {extra}
for v in ["v1_faithful", "v2_optimized"]:
    run(v, "--distinct")   # acrescente --limit 3 para testar

## Rótulos verdadeiros da causa (opcional)
O dataset só diz *interference / no-interference*. Para métricas multiclasse completas, crie `cause_labels.csv` com `scenario,cause`
(cause = um de different_superior_triangle, different_lateral_triangle, equal_superior_triangle) para os cenários com interferência. Sem esse arquivo, só `none_above` é avaliável.

In [ ]:
# 3) Métricas (binária + multiclasse) e varredura de limiar; o melhor limiar maximiza F1 e desempata por recall
LABELS = "cause_labels.csv" if os.path.exists("cause_labels.csv") else ""
for v in ["v1_faithful", "v2_optimized"]:
    flag = f"--labels {LABELS}" if LABELS else ""
    !python3 -I jev-evaluator/metrics_3sat.py results/{v} {flag}